# CUDA full-model comparison: nine attention and linear-attention variants

This is the end-to-end follow-up to `03_cuda_chunkwise_kernels.ipynb`. It benchmarks the actual four-layer GPT decoder—not only an attention operator—for:

`MHA, GQA, MQA, MLA-style, GDN, KDA, GDN ×3 + gated MHA, KDA ×3 + MLA, KDA ×3 + gated MLA`.

GDN/KDA layers use FLA chunkwise kernels for prefill/training and FLA fused recurrent kernels for cached one-token decode. MHA/GQA/MQA/MLA use the repository's PyTorch SDPA implementation. Before timing, each recurrent-containing schedule is checked against the exact reference scan.

**Scope:** random-token performance microbenchmark. Training includes embedding, positions, layer norms, token mixer, MLP, vocabulary loss, backward, and AdamW. It does not retrain models or generate new PPL values.

## 1. Use a CUDA runtime and make the repository available

In Colab select **Runtime → Change runtime type → T4 GPU** (or better). Then either clone the repository, or upload and unzip a current project archive so that `PROJECT_DIR/research/cuda_full_model_benchmark.py` exists. The source must include the new `fla` scan backend and this notebook.

In [ ]:
# FLA provides CUDA chunkwise GDN/KDA and fused recurrent decode kernels.
# If pip upgrades PyTorch, restart the runtime once and resume from this cell.
!pip -q install -U 'flash-linear-attention[cuda]==0.5.1' pandas matplotlib
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv,noheader

In [ ]:
from pathlib import Path
import os

# Option A: clone your repository here, then set PROJECT_DIR to it.
# !git clone <YOUR_REPOSITORY_URL> /content/LLMCompression
#
# Option B: upload a ZIP of the current LLMCompression folder and unzip it:
# from google.colab import files
# uploaded = files.upload()
# !unzip -q '<ARCHIVE_NAME>.zip' -d /content

PROJECT_DIR = Path('/content/LLMCompression')  # Change this if your folder has another name.
assert (PROJECT_DIR / 'research' / 'cuda_full_model_benchmark.py').is_file(), (
    'Upload/clone the current project, then set PROJECT_DIR to the folder that contains research/.')
os.chdir(PROJECT_DIR)
print('Project:', PROJECT_DIR)

In [ ]:
import sys
import torch

assert torch.cuda.is_available(), 'Enable a Colab CUDA runtime, then rerun.'
print({
    'python': sys.version.split()[0],
    'torch': torch.__version__,
    'gpu': torch.cuda.get_device_name(),
    'compute_capability': torch.cuda.get_device_capability(),
})

## 2. Run the verification gate and benchmark

The first run JIT-compiles Triton kernels; it can take several minutes on a T4. Do not interrupt `Verifying FLA equivalence` or any variant once it starts. The standard run uses FP16 on a T4, train batch 16 × 64 tokens, inference batch 8, prefill lengths 64/256/1K/4K, and 4K-prefix decode over 128 tokens.

For a smoke test set `SMOKE_TEST = True`; it produces CSVs in `results/cuda_full_model_smoke/` and is not presentation evidence.

In [ ]:
import subprocess

SMOKE_TEST = False
output_dir = 'results/cuda_full_model_smoke' if SMOKE_TEST else 'results'
command = [
    sys.executable, '-m', 'research.cuda_full_model_benchmark',
    '--output-dir', output_dir,
    '--dtype', 'auto',
]
if SMOKE_TEST:
    command += [
        '--prefill-lengths', '64', '256',
        '--decode-prompt-tokens', '256', '--decode-tokens', '16',
        '--warmup', '1', '--repeats', '2', '--verify-tokens', '16',
    ]
print(' '.join(command))
subprocess.run(command, cwd=PROJECT_DIR, check=True)

## 3. Inspect raw results

`cuda_full_model_verification.csv` must show `passed` for all five recurrent-containing schedules. The remaining four CSV files are the presentation evidence: training, prefill, decode, and environment.

In [ ]:
import pandas as pd

RESULT_DIR = PROJECT_DIR / output_dir
for filename in (
    'cuda_full_model_verification.csv',
    'cuda_full_model_training.csv',
    'cuda_full_model_prefill.csv',
    'cuda_full_model_decode.csv',
    'cuda_full_model_environment.csv',
):
    print('\n###', filename)
    display(pd.read_csv(RESULT_DIR / filename))

In [ ]:
plot_command = [
    sys.executable, '-m', 'research.plot_cuda_full_model',
    '--input-dir', output_dir, '--output-dir', output_dir,
]
subprocess.run(plot_command, cwd=PROJECT_DIR, check=True)
display(__import__('IPython').display.Image(filename=str(RESULT_DIR / 'cuda_full_model_comparison.png')))
print((RESULT_DIR / 'CUDA_FULL_MODEL_SUMMARY.md').read_text())

## 4. Download and preserve the evidence boundary

Download the five CSVs, `cuda_full_model_comparison.png`, and `CUDA_FULL_MODEL_SUMMARY.md` into the repository's `results/` directory. Do not overwrite `chunkwise_*.csv`: those are the earlier attention-operator experiment and should remain separately cited.

In [ ]:
# Optional: download files one-by-one in Colab.
# from google.colab import files
# for path in sorted(RESULT_DIR.glob('cuda_full_model_*.csv')):
#     files.download(str(path))
# files.download(str(RESULT_DIR / 'cuda_full_model_comparison.png'))
# files.download(str(RESULT_DIR / 'CUDA_FULL_MODEL_SUMMARY.md'))